# 6.4 Je eigen vectoren — wat is één rij, wanneer een rij "haha" is?

De bloemencache van 6.3 werkte omdat één rij één bloem was: een duidelijke foto, een zuiver label, één keer geëncodeerd. Chatdata biedt die luxe niet. Een groot deel van de berichten telt vijf woorden of minder, en een bericht van drie woorden embedt naar iets wat vrijwel inhoudsloos is — niet omdat de encoder slecht is, maar omdat er simpelweg nauwelijks iets is om te encoderen.

Dit notebook zet dat falen in scène, lost het op, en controleert of de oplossing daadwerkelijk werkte — eerst op de IRC-showcase, waar zes productieve auteurs de controles iets geven om tegen af te zetten, en daarna op je eigen chat, met dezelfde functies.

> Vereist `torch`, `sentence-transformers` en `transformers`: eenmalig lokaal `uv sync --extra huggingface`.

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from notebooktester import param

from goad_toolkit.visualizer import HeatmapPlot, PlotSettings, ProjectionPlot
from scripts.pipelines import build_irc_pipeline
from scripts.sessionize import fit_session_threshold, merge_messages, sessionize
from wa_analyzer.data import load_own_chat, load_showcase

EMBEDDER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


## 6.4.1 Zet de klont (blob) in scène

Zes van de meest actieve posters van #ubuntu-uk, één rij per bericht, geëmbed precies zoals ze getypt werden. Als een individueel bericht veel signaal zou dragen over wie het schreef, zou een 2D-projectie nu al zes clusters moeten tonen — dezelfde claim die 6.3 controleerde en bevestigde voor bloemenfoto's.

In [ ]:
irc = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))
uk = irc[(irc.channel == "#ubuntu-uk") & (~irc.is_action)].copy()
uk["timestamp"] = (
    pd.to_datetime(uk["date"]) + pd.to_timedelta(uk["hh"], unit="h") + pd.to_timedelta(uk["mm"], unit="m")
)
uk = uk.sort_values(["author", "timestamp"]).reset_index(drop=True)

TOP_AUTHORS = uk.author.value_counts().head(6).index.tolist()
print(f"{len(uk):,} messages, median {uk.message.str.split().str.len().median():.0f} words, "
      f"top authors: {TOP_AUTHORS}")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(42)
BLOB_SAMPLE = param(1500, test=150)
sample = uk[uk.author.isin(TOP_AUTHORS)].sample(n=BLOB_SAMPLE, random_state=42)

X = EMBEDDER.encode(sample.message.tolist(), batch_size=64, show_progress_bar=False)
coords = PCA(n_components=2).fit_transform(X)

settings = PlotSettings(
    figsize=(7, 6),
    title="1,500 individual messages, six authors — embedded one at a time",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=coords, labels=sample.author.to_numpy(),
                                        palette="tab10", s=10)

print(f"silhouette by author: {silhouette_score(X, sample.author):.3f}  (0 = no structure)")


Een vormloze klont, en een silhouette binnen afrondingsfout van nul — precies zoals het bloemenplaatje van 6.3 eruitzag *vóór* PCA, behalve dat dit *ná* embedding is. De encoder faalde niet; "thanks!" en "lol" dragen in drie woorden simpelweg niet de identiteit van zes mensen met zich mee. **Dit is een 'wat-is-één-rij'-probleem — fase 2 van de `goad_analysis_checklist`-methode, dezelfde vraag waarmee les 1 opende — niet een probleem met het model.**

## 6.4.2 Drie manieren om één rij te definiëren

Voeg opeenvolgende berichten van dezelfde auteur samen tot één eenheid, op drie verschillende manieren:

1. **Vast venster (Fixed window)** — elke 15 berichten van één auteur, waar ze toevallig ook over gaan.
2. **Per-auteur-per-dag** — alles wat één persoon op één kalenderdag zei.
3. **Tijdsinterval-sessies (Time-gap sessions)** — een nieuwe eenheid zodra het tijdsverschil sinds het vorige bericht van die persoon groter is dan een drempelwaarde. Die drempelwaarde moet niet gegokt worden.

Les 4 fitte een exponentiële verdeling op de pauzes tussen berichten, beperkt tot intervallen korter dan een uur — het regime binnen een burst — en las die fit als bewijs voor *wanneer* mensen in bursts praten in plaats van in een regelmatig tempo. Dezelfde fit, op deze data, geeft het getal dat sessievorming daadwerkelijk nodig heeft: de intervalduur waarboven een vervolg meer lijkt op toeval dan op een direct antwoord.

In [ ]:
threshold = fit_session_threshold(uk["timestamp"], uk["author"])
print(f"session threshold: {threshold:.0f}s ({threshold / 60:.1f} minutes)")

uk["session_id"] = sessionize(uk, "timestamp", "author", threshold)
subset = uk[uk.author.isin(TOP_AUTHORS)].copy()

K = 15
subset["window_id"] = subset.groupby("author").cumcount() // K
subset["day"] = subset["timestamp"].dt.date

fixed_units = merge_messages(subset, ["author", "window_id"])
daily_units = merge_messages(subset, ["author", "day"])
session_units = merge_messages(subset, ["author", "session_id"])

for name, units in [("fixed-window", fixed_units), ("per-day", daily_units), ("session", session_units)]:
    words = units.message.str.split().str.len()
    print(f"{name:>12s}: {len(units):>6,} units, median {words.median():>4.0f} words each")


~15 minuten, afgelezen van de fit in plaats van gegokt. Drie verdedigbare manieren om te antwoorden op "wat is één rij" — en de volgende sectie controleert of het antwoord er werkelijk toe doet, in plaats van alleen maar drie verschillende getallen te bewonderen.

## 6.4.3 De samenvoeging controleren, niet alleen bewonderen

Drie verdedigbare eenheden, en geen ground truth om ze tegen te scoren: handmatig geannoteerde antwoordlinks bestaan voor IRC (`jkkummerfeld/irc-disentanglement`), maar voor `#ubuntu`, niet voor de twee kanalen in de showcase van deze cursus. De controle is dus self-supervised, en heeft helemaal geen annotatie nodig: splits elke eenheid in tweeën op tijd, embed de twee helften afzonderlijk, en vraag of een nearest-neighbour-zoektocht op cosinus-similariteit de echte andere helft vaker vindt dan toeval. Een samenvoegstrategie die inhoudelijk coherente eenheden oplevert moet hier met gemak voor slagen; eentje die ruis produceert zou het niet beter moeten doen dan toeval.

In [ ]:
def self_retrieval_accuracy(frame: pd.DataFrame, group_cols: list[str], min_messages: int = 6,
                            n_eval: int = 400, seed: int = 42,
                            single_message: bool = False) -> tuple[float, int]:
    """Split each unit's messages in half by time, embed both halves, and check whether
    cosine similarity finds the true other half more often than chance.

    With `single_message=True` each half is one message rather than several joined together
    -- one from early in the unit against one from later -- which is the raw-message baseline.
    """
    sizes = frame.groupby(group_cols).size()
    keys = sizes[sizes >= min_messages].index.tolist()
    rng = np.random.default_rng(seed)
    if len(keys) > n_eval:
        keys = [keys[i] for i in rng.choice(len(keys), size=n_eval, replace=False)]

    first, second = [], []
    grouped = frame.sort_values("timestamp").groupby(group_cols)
    for key in keys:
        grp = grouped.get_group(key)
        half = len(grp) // 2
        if single_message:
            first.append(grp.message.iloc[0])
            second.append(grp.message.iloc[half])
        else:
            first.append(" ".join(grp.message.iloc[:half]))
            second.append(" ".join(grp.message.iloc[half:]))

    A = EMBEDDER.encode(first, batch_size=64, show_progress_bar=False)
    B = EMBEDDER.encode(second, batch_size=64, show_progress_bar=False)
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)
    nearest = (A @ B.T).argmax(axis=1)
    accuracy = float((nearest == np.arange(len(keys))).mean())
    return accuracy, len(keys)


Losse ruwe berichten kunnen niet doormidden gesplitst worden, dus hun baseline is dezelfde controle met `single_message=True`: één bericht van vroeg in een burst tegenover eentje van later in dezelfde burst, in plaats van een tekst die doormidden gesneden is.

In [ ]:
N_EVAL = param(400, test=40)

results = {"single message": self_retrieval_accuracy(subset, ["author", "session_id"], n_eval=N_EVAL,
                                                     single_message=True)}
results["per-day"] = self_retrieval_accuracy(subset, ["author", "day"], n_eval=N_EVAL)
results["fixed-window"] = self_retrieval_accuracy(subset, ["author", "window_id"], n_eval=N_EVAL)
results["session"] = self_retrieval_accuracy(subset, ["author", "session_id"], n_eval=N_EVAL)

print(f"{'strategy':>14s}  {'accuracy':>9s}  {'n pairs':>8s}   vs. chance")
for name, (acc, n) in results.items():
    chance = 1 / n if n else float("nan")
    print(f"{name:>14s}  {acc:>8.1%}   {n:>7,}   {acc / chance:>5.0f}x")


Elke samenvoegstrategie verslaat een enkel bericht met ruime marge, en **tijdsinterval-sessievorming wint overtuigend** — de eenheden die het produceert worden het meest betrouwbaar teruggevonden bij hun andere helft puur op basis van betekenis. Vaste vensters hakken gedachtes doormidden bij bericht 15 ongeacht wat er gebeurt; per-dag-eenheden verdunnen één echt gesprek tussen alle andere afzonderlijke gesprekken die die persoon die dag had. De grens goed krijgen, op de manier zoals §6.4.2 het fitte, is geen detail — het is het verschil tussen een bruikbare eenheid en een verdunde eenheid.

## 6.4.4 Cosinus-similariteit op samengevoegde eenheden

Twee dingen waar een samengevoegd, geëmbed corpus goed voor is: vinden wat lijkt op één specifiek ding, en vragen of "lijkt op" overeenkomt met iets wat je al kunt controleren.

In [ ]:
units = session_units[session_units.n_messages >= 5].reset_index(drop=True)
vectors = EMBEDDER.encode(units.message.tolist(), batch_size=64, show_progress_bar=False)
normed = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

rng = np.random.default_rng(7)
query = rng.integers(0, len(units))
similarity = normed @ normed[query]
ranked = np.argsort(similarity)
nearest, farthest = ranked[-2], ranked[0]  # -1 is the query itself

for label, i in [("QUERY", query), ("NEAREST", nearest), ("FARTHEST", farthest)]:
    print(f"{label} ({units.author.iloc[i]}, cosine={similarity[i]:.2f}):")
    print(f"  {units.message.iloc[i][:200]}")
    print()


De dichtstbijzijnde sessie gaat over hetzelfde soort onderwerp als de zoekopdracht — beide gaan over meer halen uit beperkt geheugen, één voor swap en één voor een GPU — en de verste is een totaal ander gesprek, over het routeren van audio over een netwerk. Cosinus-similariteit op samengevoegde eenheden zoekt op onderwerp, wat precies is wat ervan gevraagd werd en wat individuele berichten niet konden ondersteunen.

In [ ]:
centroids = pd.DataFrame(
    {author: normed[units.author.to_numpy() == author].mean(axis=0) for author in TOP_AUTHORS}
).T
centroids = centroids.div(np.linalg.norm(centroids, axis=1), axis=0)
author_similarity = pd.DataFrame(
    centroids.to_numpy() @ centroids.to_numpy().T, index=TOP_AUTHORS, columns=TOP_AUTHORS
)

settings = PlotSettings(figsize=(6, 5), title="Author centroids, cosine similarity", xlabel="", ylabel="")
fig, ax = HeatmapPlot(settings).plot(data=author_similarity, fmt=".2f")


Elk paar ligt op 0,82–0,94 — hoog, en nauwelijks onderscheidend tussen zes verschillende mensen. Dat is geen bug in de methode; het is wat een **semantische** embedding hoort te doen. Iedereen in dit kanaal bespreekt Ubuntu-ondersteuning, dus het centroïde van iedereen wijst grofweg dezelfde kant op. De dialect-vingerafdrukken uit les 5 — `:-)` tegenover `:)`, ☺ voor één persoon en niemand anders, weggelaten apostroffen — scheidden de acht drukste mensen in ditzelfde kanaal kraakhelder, maar dat signaal is *stilistisch*, gecodeerd in interpunctie en woordkeuze, niet in *betekenis*. Een embeddingmodel meet het ding waarvoor het getraind is, en dat is inhoud, geen gewoonte. Grijp naar het juiste gereedschap: trigram-tellingen voor stijl, embeddings voor onderwerp.

## 6.4.5 Laat de baseline racen tegen de embedding

6.2 bouwde een baseline die niets kost om te fitten: karakter-trigram-tellingen. Verslaat een nergens op gefinetunede sentence-embedding deze baseline op exact dezelfde samengevoegde eenheden bij de ene taak waarin hij een voordeel zou moeten hebben — zes mensen uit elkaar houden op basis van wat ze schreven?

In [ ]:
PER_AUTHOR_CAP = param(600, test=80)
balanced = units.groupby("author", group_keys=False).apply(
    lambda g: g.sample(min(len(g), PER_AUTHOR_CAP), random_state=42), include_groups=False
).join(units[["author"]])
train, test = train_test_split(balanced, test_size=0.3, stratify=balanced.author, random_state=42)

trigrams = CountVectorizer(analyzer="char", ngram_range=(3, 3), min_df=2)
Xtr_cv = trigrams.fit_transform(train.message)
Xte_cv = trigrams.transform(test.message)
acc_trigram = accuracy_score(test.author, LogisticRegression(max_iter=1000).fit(Xtr_cv, train.author).predict(Xte_cv))

Xtr_emb = EMBEDDER.encode(train.message.tolist(), batch_size=64, show_progress_bar=False)
Xte_emb = EMBEDDER.encode(test.message.tolist(), batch_size=64, show_progress_bar=False)
acc_embed = accuracy_score(test.author, LogisticRegression(max_iter=1000).fit(Xtr_emb, train.author).predict(Xte_emb))

baseline = test.author.value_counts(normalize=True).max()
print(f"majority-class baseline : {baseline:.1%}")
print(f"trigram counts + logreg : {acc_trigram:.1%}")
print(f"sentence embedding + logreg : {acc_embed:.1%}")


De goedkope, handmatig ontworpen, niet-voorgetrainde trigramvector **wint met ruime voorsprong** — beide verslaan met gemak het gokken op de meerderheidsauteur, maar het tellen van karakters identificeert deze zes mensen veel beter dan een algemene sentence-embedding doet. Dit is de bevinding uit §6.4.4 opnieuw vanuit een andere hoek: de embedding is goed in *wát er gezegd werd*, en identiteit zit hier in *hóé*. Op zes gelabelde klassen en een paar duizend rijen is het interpreteerbare model geen noodoplossing voor wanneer je het chique model niet kunt betalen — voor deze taak is het het betere instrument, en dat weet je alleen door ze tegen elkaar te laten racen.

## 6.4.6 Een sentimentscore, als afsluiter

Nog één transfer-learned model, kort: een sentiment-classifier, gedraaid op dezelfde samengevoegde sessie-eenheden. Geen projectie, geen PCA — alleen een label en een betrouwbaarheid per eenheid, geaggregeerd per auteur.

In [ ]:
from transformers import pipeline as hf_pipeline

sentiment_model = hf_pipeline(  # ty: ignore[no-matching-overload]
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    top_k=None, truncation=True, max_length=512,
)

SENTIMENT_SAMPLE = param(60, test=10)
sample_units = units.sample(n=SENTIMENT_SAMPLE, random_state=0)
scores = []
for _, row in sample_units.iterrows():
    result = {item["label"]: item["score"] for item in sentiment_model(row.message[:512])[0]}
    scores.append({"author": row.author, **result})

sentiment = pd.DataFrame(scores)
sentiment.groupby("author")[["positive", "neutral", "negative"]].mean().round(2)


Een tabel, geen claim — zestig eenheden is bij lange na niet genoeg om iets te zeggen over iemands karakter of humeur, en de "negatieve" score van een helpdesk-kanaal betreft vooral *de problemen van andere mensen*, niet het humeur van de spreker. Behandel dit zoals 5.4 een losse p-waarde behandelde: een getal, nog geen bevinding, totdat er een mechanisme en replicatie achter zit.

## 6.4.7 Jouw beurt

Dezelfde functies, je eigen chat. `fit_session_threshold` en `sessionize` weten niet en geven er niet om dat de data ooit IRC was — ze verwachten een timestamp-kolom en een author-kolom, en die heeft jouw eigen export ook.

In [ ]:
own = load_own_chat()

own_threshold = fit_session_threshold(own["timestamp"], own["author"])
print(f"your session threshold: {own_threshold:.0f}s ({own_threshold / 60:.1f} minutes)")

own = own.sort_values(["author", "timestamp"]).reset_index(drop=True)
own["session_id"] = sessionize(own, "timestamp", "author", own_threshold)
own_units = merge_messages(own, ["author", "session_id"])
checkable = own_units[own_units.n_messages >= 5]

print(f"{len(own_units)} session units, {len(checkable)} with 5+ messages")
if len(checkable) >= 20:
    acc, n = self_retrieval_accuracy(own, ["author", "session_id"], min_messages=5, n_eval=200)
    print(f"self-retrieval accuracy on your sessions: {acc:.1%} (n={n}, chance={1 / n:.1%})")
else:
    print("Not enough multi-message sessions yet for the retrieval check -- "
          "the merge itself is still the useful part.")

## De brug naar `tensordtype 2`

De ML-cursus ontmoet ditzelfde fenomeen vanuit de tegenovergestelde richting: een document dat te *lang* is voor het contextvenster van een model wordt opgeknipt in chunks, elke chunk wordt een vector, en de chunks worden weer samengevoegd tot één representatie — `(C, D) → (D)`. Dit notebook voegde berichten die te *kort* waren om iets te betekenen samen tot eenheden, embedde elke eenheid: dezelfde tensorvorm, dezelfde aggregatiestap. Lang-naar-beneden-geknipt of kort-naar-boven-samengevoegd, de bestemming is identiek: één rij, één betekenisvolle vector, en een padding-probleem dat aan de overkant op je wacht. Hetzelfde probleem, tegengestelde oorzaak — en de goedkoopste voorsprong op `tensordtype 2` van de ML-cursus.

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from wa_analyzer.data import load_own_chat
from scripts.sessionize import fit_session_threshold

# 1. Dialoogsessies bepalen op basis van de empirische pauzedrempel
own = load_own_chat()
own['dt'] = pd.to_datetime(own['timestamp'])
own = own.sort_values('dt').reset_index(drop=True)
thresh = fit_session_threshold(own['dt'], own['author'])
gaps = own['dt'].diff().dt.total_seconds().fillna(thresh + 1)
own['dialogue_id'] = (gaps > thresh).cumsum()

# 2. De 4 leefwerelden definiëren (verrijkte thematische regex-features, 16 subthema's)
domains = {
    'Voeding & Huishouden': r'\b(eten|eet|koken|boodschappen|jumbo|appie|albert heijn|lidl|brood|avondeten|lunch|lunchen|supermarkt|gegeten|pasta|pizza|koffie|restaurant|bestellen|vega)\b',
    'Huisdieren': r'\b(nusa|indy|hond|honden|hondje|hondjes|uitlaten|uitgelaten|gewandeld|wandelen|brokjes|voer|dierenarts)\b',
    'Werk & Mobiliteit': r'\b(onderweg|auto|rijden|file|trein|station|vertrokken|aankomen|hoe laat|ben thuis|werk|werken|werkt|kantoor|meeting|overleg|han|laptop|avans|moerdijk|rabobank|saskia|marit|emese|patricia|nakijken|les|bus|collega|mail|mailen)\b',
    'Zorg & Familie': r'\b(moeder|mam|mama|pap|papa|pappa|vader|ouders|schoonouders|oma|heeze|familie|verjaardag|bezoek|op bezoek|carina|sterq|buurstede|levensvreugde|zorg|zorgen|wmo|ziekenhuis|huisarts|medicijn|arts|opname)\b'
}

own['clean'] = own['message'].str.lower()
for d_name, pat in domains.items():
    own[d_name] = own['clean'].apply(lambda s: len(re.findall(pat, s)))

own['word_count'] = own['clean'].apply(lambda s: len(s.split()))

# 3. Aggregatie per dialoogsessie en per partner (gespreksbijdragen als zuivere teksteenheden)
units = own.groupby(['dialogue_id', 'author']).agg({
    'word_count': 'sum',
    **{d: 'sum' for d in domains}
}).reset_index()

units['total_theme_hits'] = sum(units[d] for d in domains)
active = units[(units['total_theme_hits'] >= 1) & (units['word_count'] >= 6)].copy()

# Relatieve woordfrequenties per 100 woorden (schaalonafhankelijk, pure tekstinhoud)
for d in domains:
    active[d + '_rate'] = (active[d] / active['word_count']) * 100.0

rate_cols = [d + '_rate' for d in domains]
X = active[rate_cols].values

# Standaardisatie naar z-scores conform les 6.1 (StandardScaler)
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# 4. Dimensiereductie: 2D PCA Biplot (geen data leakage: auteur zit NIET in featurevector X)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_std)
active['pca_x'] = coords[:, 0]
active['pca_y'] = coords[:, 1]
active['dominant_theme'] = active[rate_cols].idxmax(axis=1).str.replace('_rate', '')

var1 = pca.explained_variance_ratio_[0]
var2 = pca.explained_variance_ratio_[1]
var_total = var1 + var2


In [ ]:
# Statistische verantwoording van de zuivere 4D vectorruimte en 2D PCA biplot
ev1 = pca.explained_variance_[0]
ev2 = pca.explained_variance_[1]

print("=== METHODOLOGISCHE & STATISTISCHE VERANTWOORDING (OPDRACHT 5) ===")
print(f"Aantal inhoudelijke tekstsegmenten (N): {len(active)} (Jeffrey: {(active['author']=='Student').sum()}, Rick: {(active['author']=='Partner').sum()})")
print(f"Pauzedrempel dialoogsessies: {thresh:.0f} seconden ({thresh/60:.1f} minuten)")
print(f"Dimensies in vectorruimte: d = {len(domains)} zuivere inhoudsthema's (opgebouwd uit 16 regexes)")
print(f"Geen data leakage: auteursgegevens zitten NIET in vector X, enkel tekstinhoud.")
print()
print("=== PCA VARIANTE-ANALYSE ===")
print(f"Hoofdcomponent 1 (PC1): verklaart {var1:.1%} variantie (eigenwaarde lambda = {ev1:.2f})")
print(f"Hoofdcomponent 2 (PC2): verklaart {var2:.1%} variantie (eigenwaarde lambda = {ev2:.2f})")
print(f"Cumulatieve 2D variantie: {var_total:.1%} van alle variatie in leefwerelden")
print()
print("=== LOADINGS VAN DE 4 THEMAVECTOREN ===")
for d_name, c1, c2 in zip(domains.keys(), pca.components_[0], pca.components_[1]):
    print(f"  {d_name:24s}: PC1 = {c1:+.3f}, PC2 = {c2:+.3f}")
print()
print("=== THEMAVERDELING PER AUTEUR (CROSSTAB) ===")
ct = pd.crosstab(active['dominant_theme'], active['author'], margins=True)
ct.columns = ['Rick (Partner)', 'Jeffrey (Student)', 'Totaal']
print(ct)
print()
print("=== ZWAARTEPUNTEN (CENTROIDS) IN DE VECTORRUIMTE ===")
centroids = active.groupby('author')[['pca_x', 'pca_y']].mean()
print(f"Jeffrey (Student): PC1 = {centroids.loc['Student', 'pca_x']:+.3f}, PC2 = {centroids.loc['Student', 'pca_y']:+.3f} (neigt naar huishouden en honden)")
print(f"Rick (Partner):    PC1 = {centroids.loc['Partner', 'pca_x']:+.3f}, PC2 = {centroids.loc['Partner', 'pca_y']:+.3f} (neigt naar werk en mobiliteit)")


In [ ]:
import os
import matplotlib.pyplot as plt
from wa_analyzer.data import ROOT
from scripts.plots import JeffreyPlotSettings, apply_dav_theme, format_footnote

# Focus op titel: Voeding & Huishouden (oranje) vs. Werk & Mobiliteit (blauw)
# Context: Huisdieren en Zorg & Familie naar zachtgrijs gedempt (Grey First)
focus_themes = {
    'Voeding & Huishouden': {'color': '#D97706', 'name': 'Voeding & Huishouden'},
    'Werk & Mobiliteit':     {'color': '#2563EB', 'name': 'Werk & Mobiliteit'}
}
context_themes = ['Huisdieren', 'Zorg & Familie']

settings = JeffreyPlotSettings(figsize=(11, 9.6))
fig, ax = plt.subplots(figsize=settings.figsize)
apply_dav_theme(ax)

lim_x = 3.6
lim_y = 3.4
ax.set_xlim(-lim_x, lim_x)
ax.set_ylim(-lim_y, lim_y)

ax.axhline(0, color='#E2E8F0', linestyle='--', linewidth=1.1, zorder=1)
ax.axvline(0, color='#E2E8F0', linestyle='--', linewidth=1.1, zorder=1)

# Laag 1A: Context-punten (Huisdieren en Zorg) zachtgrijs op de achtergrond
context_mask = active['dominant_theme'].isin(context_themes)
context_df = active[context_mask]

# Jeffrey context
j_ctx = context_df[context_df['author'] == 'Student']
ax.scatter(
    j_ctx['pca_x'], j_ctx['pca_y'],
    marker='o', color='#CBD5E1', alpha=0.28, s=24, edgecolors='none', zorder=2
)
# Rick context
r_ctx = context_df[context_df['author'] == 'Partner']
ax.scatter(
    r_ctx['pca_x'], r_ctx['pca_y'],
    marker='^', color='#CBD5E1', alpha=0.28, s=26, edgecolors='none', zorder=2
)

# Laag 1B: Focus-punten (Oranje en Blauw) - zacht en transparant naar achteren
focus_markers = {
    'Student': {'marker': 'o', 'size': 28, 'alpha': 0.32},
    'Partner': {'marker': '^', 'size': 30, 'alpha': 0.32}
}

for author_key, m_info in focus_markers.items():
    sub = active[(active['author'] == author_key) & (~context_mask)]
    for theme, t_info in focus_themes.items():
        theme_sub = sub[sub['dominant_theme'] == theme]
        ax.scatter(
            theme_sub['pca_x'], theme_sub['pca_y'],
            marker=m_info['marker'],
            color=t_info['color'],
            alpha=m_info['alpha'],
            s=m_info['size'],
            edgecolors='none',
            zorder=3
        )

# Laag 2: Zwaartepunten (Centroids) van beide partners
j_center = active[active['author'] == 'Student'][['pca_x', 'pca_y']].mean()
r_center = active[active['author'] == 'Partner'][['pca_x', 'pca_y']].mean()

ax.scatter(
    j_center['pca_x'], j_center['pca_y'],
    marker='o', s=130, facecolor='#FFFFFF', edgecolor='#D97706', linewidth=2.5,
    zorder=5
)
ax.scatter(
    j_center['pca_x'], j_center['pca_y'],
    marker='o', s=35, color='#D97706', zorder=6
)
ax.annotate(
    "Zwaartepunt Jeffrey\n(-0.14, +0.04)",
    xy=(j_center['pca_x'], j_center['pca_y']),
    xytext=(-15, 14), textcoords='offset points',
    fontsize=8.5, fontweight='bold', color='#B45309', ha='right', va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='#FEF3C7', edgecolor='#F59E0B', alpha=0.92, lw=1.0),
    zorder=7
)

ax.scatter(
    r_center['pca_x'], r_center['pca_y'],
    marker='^', s=130, facecolor='#FFFFFF', edgecolor='#2563EB', linewidth=2.5,
    zorder=5
)
ax.scatter(
    r_center['pca_x'], r_center['pca_y'],
    marker='^', s=35, color='#2563EB', zorder=6
)
ax.annotate(
    "Zwaartepunt Rick\n(+0.16, -0.04)",
    xy=(r_center['pca_x'], r_center['pca_y']),
    xytext=(18, 18), textcoords='offset points',
    fontsize=8.5, fontweight='bold', color='#1D4ED8', ha='left', va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='#EFF6FF', edgecolor='#3B82F6', alpha=0.92, lw=1.0),
    zorder=7
)

# Laag 3: De 4 thematische PCA-vectoren (Loadings)
scale_arrow = 3.5

arrow_styles = {
    'Voeding & Huishouden': {'color': '#D97706', 'lw': 2.8, 'alpha': 1.0, 'bold': True, 'offset': (-14, -14), 'ha': 'right', 'va': 'top'},
    'Werk & Mobiliteit':     {'color': '#2563EB', 'lw': 2.8, 'alpha': 1.0, 'bold': True, 'offset': (14, -4),   'ha': 'left',  'va': 'center'},
    'Huisdieren':            {'color': '#94A3B8', 'lw': 1.6, 'alpha': 0.85, 'bold': False, 'offset': (-14, 8),  'ha': 'right', 'va': 'bottom'},
    'Zorg & Familie':        {'color': '#94A3B8', 'lw': 1.6, 'alpha': 0.85, 'bold': False, 'offset': (12, 10),  'ha': 'left',  'va': 'bottom'},
}

for col, c1, c2 in zip(domains.keys(), pca.components_[0], pca.components_[1]):
    vx = c1 * scale_arrow
    vy = c2 * scale_arrow
    astyle = arrow_styles[col]
    
    ax.annotate(
        '', xy=(vx, vy), xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color=astyle['color'], lw=astyle['lw'], alpha=astyle['alpha'], mutation_scale=18),
        zorder=6 if astyle['bold'] else 3
    )
    
    ox, oy = astyle['offset']
    fontweight = 'bold' if astyle['bold'] else 'normal'
    fontsize = 9.5 if astyle['bold'] else 8.2
    
    ax.annotate(
        col, xy=(vx, vy), xytext=(ox, oy), textcoords='offset points',
        fontsize=fontsize, fontweight=fontweight, color=astyle['color'], ha=astyle['ha'], va=astyle['va'],
        bbox=dict(
            boxstyle='round,pad=0.25',
            facecolor='#FFFFFF',
            edgecolor=astyle['color'] if astyle['bold'] else '#CBD5E1',
            alpha=0.94,
            lw=1.2 if astyle['bold'] else 0.8
        ),
        zorder=7 if astyle['bold'] else 4
    )

# Titels & Header: direct en krachtig, geen 'Figuur 5:'
title_text = "Jeffrey is meer de huisman, terwijl Rick zich focust op werk in onze chat"
subtitle_text = "2D PCA-biplot (62.4% variantie). Rick (▲) domineert werk & mobiliteit (blauw); Jeffrey (●) domineert huishouden (oranje). Overige contexten (honden en zorg) zijn grijs gedempt."
ax.set_title(title_text, fontsize=12.2, fontweight='bold', pad=28, color='#1A202C', loc='left')
ax.text(0.0, 1.02, subtitle_text, transform=ax.transAxes, fontsize=9.2, style='italic', color='#4A5568', va='bottom', ha='left')

ax.set_xlabel(f"Hoofdcomponent 1 ({var1:.1%} variantie; λ = 1.37) — Huishouden/Honden (links) vs. Werk/Mobiliteit (rechts)", fontsize=9.2, color='#1A202C')
ax.set_ylabel(f"Hoofdcomponent 2 ({var2:.1%} variantie; λ = 1.13) — Familie & Mantelzorg (boven)", fontsize=9.2, color='#1A202C')

# Cijfers voor 4D-verdelingstabel in de voetnoot
n_total = len(active)
ct = pd.crosstab(active['dominant_theme'], active['author'])
r_werk = ct.loc['Werk & Mobiliteit', 'Partner']
j_werk = ct.loc['Werk & Mobiliteit', 'Student']
tot_werk = r_werk + j_werk

r_voed = ct.loc['Voeding & Huishouden', 'Partner']
j_voed = ct.loc['Voeding & Huishouden', 'Student']
tot_voed = r_voed + j_voed

r_hond = ct.loc['Huisdieren', 'Partner']
j_hond = ct.loc['Huisdieren', 'Student']
tot_hond = r_hond + j_hond

r_zorg = ct.loc['Zorg & Familie', 'Partner']
j_zorg = ct.loc['Zorg & Familie', 'Student']
tot_zorg = r_zorg + j_zorg

footnote_text = (
    f"* Verdeling tekstsegmenten (N = {n_total}, d = 4, pauzedrempel = {thresh/60:.0f} min):\n"
    f"  Werk & Mobiliteit:      {tot_werk} segmenten  |  Rick: {r_werk:3d} ({r_werk/tot_werk:.0%})  vs.  Jeffrey: {j_werk:3d} ({j_werk/tot_werk:.0%})\n"
    f"  Voeding & Huishouden:   {tot_voed} segmenten  |  Rick: {r_voed:3d} ({r_voed/tot_voed:.0%})  vs.  Jeffrey: {j_voed:3d} ({j_voed/tot_voed:.0%})\n"
    f"  Huisdieren:             {tot_hond} segmenten  |  Rick: {r_hond:3d} ({r_hond/tot_hond:.0%})  vs.  Jeffrey: {j_hond:3d} ({j_hond/tot_hond:.0%})\n"
    f"  Zorg & Familie:          {tot_zorg} segmenten  |  Rick: {r_zorg:3d} ({r_zorg/tot_zorg:.0%})  vs.  Jeffrey: {j_zorg:3d} ({j_zorg/tot_zorg:.0%})\n"
    f"* Methodologie: Zuivere 4D vectorruimte opgebouwd uit 16 regexes (z-scores via StandardScaler).\n"
    f"  2D PCA verklaart {var_total:.1%} van de totale variantie (PC1: {var1:.1%}, PC2: {var2:.1%}; PC3: 19.3%, PC4: 18.3%).\n"
    f"  Zwaartepunten: Jeffrey ({j_center['pca_x']:+.2f}, {j_center['pca_y']:+.2f}), Rick ({r_center['pca_x']:+.2f}, {r_center['pca_y']:+.2f})."
)
format_footnote(ax, text=footnote_text, y_offset=-0.25)

# Volledig Schwabish/Tufte: geen losse legenda, subtitel verklaart direct de visuele codering

plt.subplots_adjust(bottom=0.29, top=0.84, left=0.10, right=0.95)

out_fig5 = os.path.join(ROOT, 'img', 'figuur5_vectorruimte_gesprek.png')
fig.savefig(out_fig5, dpi=300, bbox_inches='tight')
print(f'SUCCESS: Figuur 5 opgeslagen in {out_fig5}')
plt.show()
